# Experimento 12.00 — Auditoria inicial do acervo

**Não treina modelos, não coleta dados da internet, não modifica os arquivos existentes.** Registra cobertura, esquema, hashes e possíveis duplicações de cópias do projeto.

In [1]:
from pathlib import Path
import hashlib, json, datetime
import pandas as pd
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass
ROOT=Path('/content/drive/MyDrive')
CANDIDATAS=sorted(p for p in ROOT.iterdir() if p.is_dir() and p.name.startswith('mixers-ethereum'))
for i,p in enumerate(CANDIDATAS):
    print(i, p, 'treino:', (p/'data/processed/features_treino.parquet').exists())
ESCOLHA_INDICE=None # configure explicitamente se mais de uma pasta; nunca escolha automaticamente
if len(CANDIDATAS)>1 and ESCOLHA_INDICE is None:
    raise RuntimeError('Há mais de uma cópia; informe ESCOLHA_INDICE antes de prosseguir.')
BASE=CANDIDATAS[ESCOLHA_INDICE or 0]
print('Pasta selecionada:', BASE)


Mounted at /content/drive
0 /content/drive/MyDrive/mixers-ethereum treino: True
1 /content/drive/MyDrive/mixers-ethereum (1) treino: True


RuntimeError: Há mais de uma cópia; informe ESCOLHA_INDICE antes de prosseguir.

In [ ]:
ARQUIVOS=[
 'data/raw/blockchair_ethereum_transactions_20201029.tsv.gz',
 'data/raw/blockchair_ethereum_transactions_20250315.tsv.gz',
 'data/raw/blockchair_ethereum_transactions_20260115.tsv.gz',
 'data/interim/test_20201029.parquet',
 'data/interim/dia_20250315.parquet',
 'data/interim/dia_20260115.parquet',
 'data/interim/tornado10_2025_03.parquet',
 'data/processed/features_teste_20201029.parquet',
 'data/processed/features_treino.parquet',
 'data/processed/features_2026-01-15.parquet',
 'data/processed/modelo_rf.joblib',
]
def hash_file(path):
    h=hashlib.sha256()
    with path.open('rb') as f:
        for b in iter(lambda:f.read(4*1024*1024), b''): h.update(b)
    return h.hexdigest()
rows=[]
for rel in ARQUIVOS:
    p=BASE/rel
    row={'arquivo':rel,'existe':p.is_file(),'tamanho_bytes':p.stat().st_size if p.is_file() else None,
         'sha256':hash_file(p) if p.is_file() else None}
    rows.append(row)
inventario=pd.DataFrame(rows)
display(inventario)


,arquivo,existe,tamanho_bytes,sha256
0,data/raw/blockchair_ethereum_transactions_2020...,True,239986765,4acdd008e41ec368755bba14858d2f414a8e79e1c17290...
1,data/raw/blockchair_ethereum_transactions_2025...,True,293598528,01e6037103e2cad295a49b1c76eddf36676d68bb475f1f...
2,data/raw/blockchair_ethereum_transactions_2026...,True,752673845,0541655b96dbaed92cb7a40e01eb44293f9d27a8efb247...
3,data/interim/test_20201029.parquet,True,141354703,0cf78a4903a94edefd76eb09359e420c378b5660990a57...
4,data/interim/dia_20250315.parquet,True,132322464,23fa755eedfdf2ad36fd86f7de696870c97f7cafa35c65...
5,data/interim/dia_20260115.parquet,True,369367413,9537f05d29044991f9a4750de1ea1b2dc5a466c4155826...
6,data/interim/tornado10_2025_03.parquet,True,712270,4e7ac5c2c65c2a772f86b8ae5bb37341a712195d3a590b...
7,data/processed/features_teste_20201029.parquet,True,9217930,6f8fd4e83e816c785d449116699065ad77425c9da6e7b1...
8,data/processed/features_treino.parquet,True,13025,0a8baa11cee97d338d7a8d92b984f38192ba1a10ec915c...
9,data/processed/features_2026-01-15.parquet,True,12591717,6d3da4a245778d707319cec8cde7a420a91166f01cd637...


In [ ]:
# Leitura apenas de cabeçalhos dos parquets processados, sem carregar dumps brutos na RAM
from importlib.util import find_spec
if find_spec('pyarrow') is None:
    raise RuntimeError('Instale pyarrow em uma célula separada: %pip install pyarrow')
import pyarrow.parquet as pq
esquemas=[]
for rel in ARQUIVOS:
    p=BASE/rel
    if p.exists() and p.suffix=='.parquet':
        meta=pq.ParquetFile(p)
        esquemas.append({'arquivo':rel,'linhas':meta.metadata.num_rows,
                         'colunas':list(meta.schema.names)})
display(pd.DataFrame(esquemas))


,arquivo,linhas,colunas
0,data/interim/test_20201029.parquet,1114702,"[hash, time, failed, type, sender, recipient, ..."
1,data/interim/dia_20250315.parquet,1009770,"[hash, time, failed, sender, recipient, call_c..."
2,data/interim/dia_20260115.parquet,2819294,"[hash, time, failed, sender, recipient, call_c..."
3,data/interim/tornado10_2025_03.parquet,994,"[block_id, id, index, hash, date, time, failed..."
4,data/processed/features_teste_20201029.parquet,230658,"[recipient, dia, n_tx, value_entropy, fee_entr..."
5,data/processed/features_treino.parquet,123,"[recipient, dia, n_tx, value_entropy, fee_entr..."
6,data/processed/features_2026-01-15.parquet,302716,"[recipient, dia, n_tx, value_entropy, fee_entr..."


In [ ]:
# Guardar SOMENTE relatório da auditoria em nova pasta versionada
stamp=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
out=BASE/'temporal_2020_2026'/'results'/('auditoria_'+stamp)
out.mkdir(parents=True,exist_ok=False)
inventario.to_csv(out/'inventario_sha256.csv',index=False)
pd.DataFrame(esquemas).assign(colunas=lambda d:d.colunas.map(json.dumps)).to_csv(out/'esquemas_parquet.csv',index=False)
(out/'README.md').write_text('Auditoria de inventário. Nenhum modelo treinado e nenhum arquivo original alterado.\\n')
print('Auditoria salva em:',out)


Auditoria salva em: /content/drive/MyDrive/mixers-ethereum/temporal_2020_2026/results/auditoria_20261115T011054Z
